In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from datetime import *


# Projects DataFrame
projects_data = [
    (1, "Skyscraper", "2022-01-01", "2022-12-31", 15000000),
    (2, "Bridge", "2022-03-01", "2022-08-31", 5000000),
    (3, "Tunnel", "2022-06-01", "2023-01-31", 10000000),
]

projects_columns = ["project_id", "project_name", "start_date", "end_date", "budget"]

projects = spark.createDataFrame(projects_data, projects_columns)


employees_data = [
    (1, "John", "Doe", "Engineer", 1),
    (2, "Jane", "Smith", "Architect", 1),
    (3, "Jim", "Brown", "Project Manager", 1),
    (4, "Emily", "Davis", "Engineer", 2),
    (5, "Alan", "Johnson", "Architect", 2),
]

employees_columns = ["employee_id", "first_name", "last_name", "role", "project_id"]

employees = spark.createDataFrame(employees_data, employees_columns)


equipment_data = [
    (1, "Crane", 1, 25000),
    (2, "Excavator", 1, 15000),
    (3, "Bulldozer", 2, 20000),
    (4, "Loader", 2, 10000),
    (5, "Crane", 3, 25000),
]

equipment_columns = ["equipment_id", "equipment_name", "project_id", "cost"]

equipment = spark.createDataFrame(equipment_data, equipment_columns)


projects=projects.withColumn('duration_days',date_diff(to_date("end_date"), to_date("start_date")))

projects.display()

employees_agg = employees.groupBy(
        "project_id"
    ).agg(
        count("employee_id").alias(
            "total_employees"
        ),
        countDistinct("role").alias(
            "unique_roles"
        ),
    )

employees_agg.display()


equipment_agg = equipment.groupBy(
        "project_id"
    ).agg(
        sum("cost").alias(
            "total_equipment_cost"
        )
    )

project_summary = projects.join(employees_agg,"project_id","left")\
        .join(equipment_agg,"project_id","left")\
        .select(
            "project_id",
            "project_name",
            "start_date",
            "end_date",
            "duration_days",
            "total_employees",
            "unique_roles",
            "total_equipment_cost"
        )
    
project_summary.display()


